## Проверка результатов подбора параметров с помощью optuna

In [ ]:
import json

with open('/wrk/main/optuna_padim_18.json', 'r') as f:
    data = json.load(f)

filtered_data = {key: value for key, value in data.items() if "error" not in value}

data_items = sorted(filtered_data.items(), key=lambda x: x[1]["metrics"]['image_F1Score'], reverse=True)

data_items[:2]

Пример вывода данных:

[('120',  
  {'trial': 120,  
   'params': {'layers': ['layer3'],  
    'n_features': 450,  
    'sensitivity': 0.5083554678816509,  
    'resize_size': 256},  
   'metrics': {'image_Accuracy': 0.9215436706550598,  
    'image_Precision': 0.9289504330265808,  
    'image_Recall': 0.9143215412216187,  
    'image_F1Score': 0.9201234503283691,  
    'image_AUROC': 0.9214323506550598},  
   'result': 0.9201234503283691}),  
    
 ('96',   
  {'trial': 96,  
   'params': {'layers': ['layer2', 'layer3'],  
    'n_features': 370,  
    'sensitivity': 0.4993486276156505,  
    'resize_size': 256},  
   'metrics': {'image_Accuracy': 0.9184392412216187,  
    'image_Precision': 0.9048943214385376,  
    'image_Recall': 0.9251233336677551,  
    'image_F1Score': 0.9144737560992432,  
    'image_AUROC': 0.9143356816169739},  
   'result': 0.9144737560992432})]  

In [ ]:
import json

with open('/wrk/main/optuna_padim_50.json', 'r') as f:
    data = json.load(f)

filtered_data = {key: value for key, value in data.items() if "error" not in value}

data_items = sorted(filtered_data.items(), key=lambda x: x[1]["metrics"]['image_F1Score'], reverse=True)

data_items[:2]

## Пример воспроизведения обучения с наилучшими параметрами

In [ ]:
from anomalib.engine import Engine
import torch
from anomalib.models.image.padim import Padim
from torchvision import models
import os
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize

torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')
seed_everything(42, workers=True)

# -----------------
#      Metrics
# -----------------

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC])

post_processor = PostProcessor(
    image_sensitivity=0.5083554678816509,
    pixel_sensitivity=0.5083554678816509
)

pre_processor = PreProcessor(transform=Resize(size=(256,256)))

# -----------------
#       Data
# -----------------

train_dataset = Folder(
    name="main",
    root="/wrk/main/data",
    normal_dir="train",
    abnormal_dir="defect_test",
    #mask_dir="masks",
    normal_test_dir="normal_test",
    train_batch_size=10,
    eval_batch_size=10,
    num_workers=4,
    seed=42
)

# -----------------
#      Model
# -----------------

weights = torch.load("/wrk/CNN_weights/resnet18.pth", map_location='cuda')
custom_backbone = models.resnet18()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

if custom_backbone:
    print("Local weights loaded successfully")

model = Padim(
    backbone=custom_backbone,
    layers=['layer3'],
    n_features=450,
    pre_trained=False,
    evaluator=evaluator,
    post_processor=post_processor,
    pre_processor=pre_processor
)

device = 'cuda'
model = model.to(device)

engine = Engine(
                accelerator='cuda', 
                enable_progress_bar=True
                )

engine.train(model=model, datamodule=train_dataset)

print(engine.trainer.callback_metrics)